## Linking without deduplication

With `link_only`, only between-dataset record comparisons are generated. No within-dataset record comparisons are created, meaning that the model does not attempt to find within-dataset duplicates.


In [ ]:
# Uncomment and run this cell if you're running in Google Colab.
# !pip install splink

In [1]:
from splink import splink_datasets

df = splink_datasets.fake_1000

# Split a simple dataset into two, separate datasets which can be linked together.
df_l = df.sample(frac=0.5)
df_r = df.drop(df_l.index)

df_l.head(2)

,unique_id,first_name,surname,dob,city,email,cluster
202,202,NaN,Taylor,1984-10-27,Huddersfield,g.t51@rodriguez.com,54
86,86,Charlotte,Johnson,2012-01-06,fTelford,charlottej68@lee-taylor@.org,25


In [2]:
import splink.comparison_library as cl

from splink import DuckDBAPI, Linker, SettingsCreator, block_on

settings = SettingsCreator(
    link_type="link_only",
    blocking_rules_to_generate_predictions=[
        block_on("first_name"),
        block_on("surname"),
    ],
    comparisons=[
        cl.NameComparison(
            "first_name",
        ),
        cl.NameComparison("surname"),
        cl.DateOfBirthComparison(
            "dob",
            input_is_string=True,
            invalid_dates_as_null=True,
        ),
        cl.ExactMatch("city").configure(term_frequency_adjustments=True),
        cl.EmailComparison("email"),
    ],
)

linker = Linker(
    [df_l, df_r],
    settings,
    db_api=DuckDBAPI(),
    input_table_aliases=["df_left", "df_right"],
)

In [3]:
from splink.exploratory import completeness_chart

completeness_chart(
    [df_l, df_r],
    cols=["first_name", "surname", "dob", "city", "email"],
    db_api=DuckDBAPI(),
    table_names_for_chart=["df_left", "df_right"],
)

alt.LayerChart(...)

In [4]:

deterministic_rules = [
    "l.first_name = r.first_name and levenshtein(r.dob, l.dob) <= 1",
    "l.surname = r.surname and levenshtein(r.dob, l.dob) <= 1",
    "l.first_name = r.first_name and levenshtein(r.surname, l.surname) <= 2",
    block_on("email"),
]


linker.training.estimate_probability_two_random_records_match(deterministic_rules, recall=0.7)

Probability two random records match is estimated to be  0.00319.
This means that amongst all possible pairwise record comparisons, one in 313.06 are expected to match.  With 250,000 total possible comparisons, we expect a total of around 798.57 matching pairs


In [5]:
linker.training.estimate_u_using_random_sampling(max_pairs=1e6, seed=1)

You are using the default value for `max_pairs`, which may be too small and thus lead to inaccurate estimates for your model's u-parameters. Consider increasing to 1e8 or 1e9, which will result in more accurate estimates, but with a longer run time.
----- Estimating u probabilities using random sampling -----

Estimated u probabilities using random sampling

Your model is not yet fully trained. Missing estimates for:
    - first_name (no m values are trained).
    - surname (no m values are trained).
    - dob (no m values are trained).
    - city (no m values are trained).
    - email (no m values are trained).


In [6]:
session_dob = linker.training.estimate_parameters_using_expectation_maximisation(block_on("dob"))
session_email = linker.training.estimate_parameters_using_expectation_maximisation(
    block_on("email")
)
session_first_name = linker.training.estimate_parameters_using_expectation_maximisation(
    block_on("first_name")
)


----- Starting EM training session -----

Estimating the m probabilities of the model by blocking on:
l."dob" = r."dob"

Parameter estimates will be made for the following comparison(s):
    - first_name
    - surname
    - city
    - email

Parameter estimates cannot be made for the following comparison(s) since they are used in the blocking rules: 
    - dob

Level Jaro-Winkler >0.88 on username on comparison email not observed in dataset, unable to train m value

Iteration 1: Largest change in params was -0.407 in the m_probability of email, level `Exact match on email`
Iteration 2: Largest change in params was 0.119 in probability_two_random_records_match
Iteration 3: Largest change in params was 0.0566 in the m_probability of first_name, level `All other comparisons`
Iteration 4: Largest change in params was 0.0202 in probability_two_random_records_match
Iteration 5: Largest change in params was 0.00861 in probability_two_random_records_match
Iteration 6: Largest change in params

In [7]:
results = linker.inference.predict(threshold_match_probability=0.9)

Blocking time: 0.01 seconds
Predict time: 0.19 seconds


In [8]:
results.as_pandas_dataframe(limit=5)

,match_weight,match_probability,source_dataset_l,source_dataset_r,unique_id_l,unique_id_r,first_name_l,first_name_r,gamma_first_name,surname_l,...,dob_l,dob_r,gamma_dob,city_l,city_r,gamma_city,email_l,email_r,gamma_email,match_key
0,15.737846,0.999982,df_left,df_right,975,972,Leon,Leon,4,Armstronog,...,1980-04-08,1970-04-11,1,Salford,Salford,1,l.armstrong@reyes-campbell.net,l.armstrong@reyes-campbell.net,4,0
1,21.829011,1.000000,df_left,df_right,43,37,Theodore,Theodore,4,Morris,...,1978-08-19,1978-08-19,5,Birmingham,Birmingham,1,t.m39@brooks-sawyer.com,t.m39@brooks-sawyer.com,4,0
2,22.775204,1.000000,df_left,df_right,949,947,Sarah,Sarah,4,Pearson,...,2011-01-07,2011-01-07,5,London,London,1,sarahp77@humphrey.com,sarahp77@humphrey.com,4,0
3,16.146929,0.999986,df_left,df_right,670,671,Ollie,Ollie,4,Rowe,...,2006-12-05,2006-12-05,5,Manchester,Manchester,1,or@cox-blackwell.org,or@cox-blackwell.org,4,0
4,13.401063,0.999908,df_left,df_right,9,10,Evie,None,-1,Dean,...,2015-03-03,2015-03-03,5,Pootsmruth,Portsmouth,0,evihd56@earris-bailey.net,evied56@harris-bailey.net,2,1
